# RSA Cryptography

**Course**: Quantum Computing Laboratory  
**Version**: 2.2 (Qiskit 2.2 Compatible)  
**Last Updated**: November 2025

---

## 📚 Theory and Background

**RSA (Rivest–Shamir–Adleman)** is an asymmetric cryptographic algorithm used for secure data transmission. It's called 'asymmetric' because it uses two different keys:

- **Public Key**: Can be shared openly, used for encryption
- **Private Key**: Must be kept secret, used for decryption

### Security Foundation

RSA's security relies on the **computational difficulty of factoring large integers**:  
- Easy to multiply two large primes: p × q = N
- Very hard to factor N back to p and q
- **Classical difficulty**: O(n^(1/4)(log(n))^3)) Number Field Sieve - Pollard 1988

### Why This Matters for Quantum Computing

**Shor's Algorithm** can factor integers in polynomial time on a quantum computer:  
- **Quantum complexity**: O((ln N)³) - polynomial time
- **Threat to RSA**: Large-scale quantum computers could break current RSA encryption
- **Post-quantum cryptography**: New algorithms resistant to quantum attacks

### Applications

- Secure web browsing (HTTPS)
- Email encryption (PGP/GPG)
- Digital signatures
- Cryptocurrency
- Secure file transfer

## 🎯 Learning Objectives

1. ✅ Understand RSA key generation
2. ✅ Implement RSA encryption and decryption
3. ✅ Learn Euler's theorem and modular arithmetic
4. ✅ Understand the quantum threat to RSA
5. ✅ Explore post-quantum alternatives

## 📐 Mathematical Foundation

### Modular Arithmetic

The modulo operation finds the remainder after division:
$$4 \div 3 = 1 \text{ remainder } 1 \quad \Rightarrow \quad 4 \equiv 1 \pmod{3}$$

$$7 \div 3 = 2 \text{ remainder } 1 \quad \Rightarrow \quad 7 \equiv 1 \pmod{3}$$

Therefore: $4 \equiv 7 \pmod{3}$

### Euler's Totient Function

For N = p × q where p, q are prime:
$$\phi(N) = (p-1)(q-1)$$

### Euler's Theorem

For any a < N where gcd(a, N) = 1 (a and N are "coprime"):
$$a^{\phi(N)} \equiv 1 \pmod{N}$$

This is the foundation of RSA!

### Key Generation

1. Choose two large primes p and q
2. Compute N = p × q
3. Compute φ(N) = (p-1)(q-1)
4. Choose e where: 1 < e < φ(N) and gcd(e, φ(N)) = 1
5. Find d where: e × d ≡ 1 (mod φ(N))
6. **Public key**: (e, N)
7. **Private key**: (d, N)

### Encryption/Decryption

**Encrypt** (with public key):
$$c = m^e \mod N$$

**Decrypt** (with private key):
$$m = c^d \mod N$$

**Why it works**:
$$c^d = (m^e)^d = m^{ed} = m^{1 + k\phi(N)} = m \cdot (m^{\phi(N)})^k \equiv m \cdot 1^k \equiv m \pmod{N}$$

In [1]:
# Classical RSA implementation
import math            # math.gcd = Euclid's algorithm, the only piece of number theory we take from the library
import random          # not used below: handy for random messages or candidate primes in the exercises
import numpy as np     # not used below: imported for uniformity with the other notebooks

print("✓ Imports successful!")
print("  Python math and random modules loaded")


✓ Imports successful!
  Python math and random modules loaded


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `import math` | `math.gcd(a, b)` is Euclid's algorithm: the one library function the key generation needs |
| `random`, `numpy` | imported but unused: this is a plain-Python notebook — no Qiskit, no circuits, nothing runs on a quantum backend (which is why the IQM edition is identical) |

</details>


## 🔧 Implementation

### Step 1: Modular Arithmetic Examples

In [2]:
# Demonstrate modular arithmetic in Python

# Modulo operator: %
print("Basic Modulo Operations:")
print(f"  5 % 4 = {5 % 4}  (remainder when 5 divided by 4)")              # f-string: the expression inside {} is evaluated and inserted
print(f"  9 % 4 = {9 % 4}  (remainder when 9 divided by 4)")
print(f"  Therefore: 5 ≡ 9 (mod 4) is {5 % 4 == 9 % 4}")                  # congruent = same remainder; == gives a bool, printed as True/False

# Modular exponentiation: pow(base, exponent, modulus)
print("\nModular Exponentiation:")                                         # \n = a blank line before the heading
print(f"  2^5 mod 7 = {pow(2, 5, 7)}")                                     # 3-argument pow: squares and reduces mod 7 at every step
print(f"  This is much more efficient than (2**5) % 7 for large numbers!")   # 2**5 % 7 first builds 2**5; with a 2048-bit exponent that number would never fit in memory


Basic Modulo Operations:
  5 % 4 = 1  (remainder when 5 divided by 4)
  9 % 4 = 1  (remainder when 9 divided by 4)
  Therefore: 5 ≡ 9 (mod 4) is True

Modular Exponentiation:
  2^5 mod 7 = 4
  This is much more efficient than (2**5) % 7 for large numbers!


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `a % n` | the remainder of the integer division: the 'mod' of modular arithmetic; two numbers are congruent mod n when their remainders agree |
| `pow(a, m, n)` | aᵐ mod n by repeated squaring with a reduction at every step: the numbers never exceed n², so it works for 2048-bit exponents |
| `f"... {5 % 4 == 9 % 4}"` | any expression can sit inside the braces of an f-string, here a comparison that prints as True |

</details>


In [3]:
2**5 % 7        # ** binds tighter than %: (2**5) % 7 = 32 % 7 = 4; the value of a cell's last expression is shown as its output


4

<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `2**5 % 7` | `**` is evaluated before `%`: 32 mod 7 = 4; the big intermediate number 2⁵ is built in full |
| bare expression | a cell whose last line is an expression shows its value as the output, without print |

</details>


In [4]:
pow(2,5,7)      # same result, computed with intermediate reductions: 2, 4, 8→1, 2, 4 (mod 7)


4

<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `pow(2, 5, 7)` | the same 4, but reduced mod 7 after each multiplication; for `pow(2, 10**6, 7)` this is the difference between instant and impossible |

</details>


### Step 2: Helper Functions

In [5]:
def euler_totient(p: int, q: int) -> int:        # type hints (p: int, -> int) document the types; Python does not enforce them
    """
    Calculate Euler's totient function φ(N) for N = p×q.
    
    Args:
        p: First prime number
        q: Second prime number
    
    Returns:
        φ(N) = (p-1)(q-1)
    
    Raises:
        AssertionError: If p and q are not coprime
    """
    assert math.gcd(p, q) == 1, "p and q must be coprime (relatively prime)"   # assert stops with the message if the condition is False
    return (p - 1) * (q - 1)                      # φ(pq) = (p−1)(q−1): how many 1 ≤ a < N are coprime with N


def modular_inverse(e: int, phi: int) -> int:
    """
    Find modular multiplicative inverse d where: e × d ≡ 1 (mod φ).
    
    Brute-force search over all candidates (the In-depth analysis at the end
    replaces it with the extended Euclidean algorithm).
    
    Args:
        e: Public exponent
        phi: Euler's totient φ(N)
    
    Returns:
        d: Private exponent where e×d ≡ 1 (mod φ)
    """
    e = e % phi                                   # reduce e first: only its remainder mod φ matters
    for x in range(1, phi):                       # try x = 1, 2, …, φ−1 in turn: up to φ steps — fine for φ = 3120, impossible for a 2048-bit key
        if (e * x) % phi == 1:                    # the defining property of the inverse
            return x                              # the first hit is the inverse (unique mod φ)
    return 1                                      # no inverse exists (gcd(e, φ) ≠ 1); a real implementation would raise an error here


def encrypt(message: int, e: int, n: int) -> int:
    """
    Encrypt message using RSA public key.
    
    Args:
        message: Plaintext message (as integer)
        e: Public exponent
        n: Modulus (N = p×q)
    
    Returns:
        ciphertext: Encrypted message
    """
    return pow(message, e, n)                     # c = m^e mod N, by repeated squaring


def decrypt(ciphertext: int, d: int, n: int) -> int:
    """
    Decrypt ciphertext using RSA private key.
    
    Args:
        ciphertext: Encrypted message
        d: Private exponent
        n: Modulus (N = p×q)
    
    Returns:
        plaintext: Decrypted message
    """
    return pow(ciphertext, d, n)                  # m = c^d mod N: the same operation with the private exponent


print("✓ Helper functions defined")


✓ Helper functions defined


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `def f(p: int, q: int) -> int:` | type hints: documentation for the reader (and for tools), not enforced at run time |
| `assert cond, msg` | a guard: stops with `msg` if the condition is False; distinct primes are always coprime, so it only catches p = q |
| `(p - 1) * (q - 1)` | Euler's totient of N = pq: the number of integers below N that are coprime with N — the size of the group in which Euler's theorem lives |
| `for x in range(1, phi): if (e * x) % phi == 1` | brute-force inverse: tries every candidate — 2753 steps here, ~10⁶¹⁶ for a real key (the In-depth analysis uses Euclid instead); the docstring of the 2025 notebook claimed 'extended Euclid' — it is not |
| `encrypt` / `decrypt` | the same `pow(x, exponent, n)` with the public exponent e or the private exponent d: RSA is symmetric in its two exponents |

</details>


### Step 3: RSA Key Generation

In [6]:
# Choose two prime numbers
# (In practice, these would be hundreds of digits long!)
p = 61
q = 53

print("Step 1: Choose primes")
print(f"  p = {p}")
print(f"  q = {q}")

# Compute N = p × q
N = p * q                                        # the public modulus; factoring it back into p and q is the hard problem RSA rests on
print(f"\nStep 2: Compute N = p × q")
print(f"  N = {p} × {q} = {N}")

# Compute Euler's totient
phi_N = euler_totient(p, q)                      # φ(N) = 60 × 52 = 3120: secret, because computing it needs p and q
print(f"\nStep 3: Compute φ(N) = (p-1)(q-1)")
print(f"  φ({N}) = ({p}-1)×({q}-1) = {phi_N}")

# Choose public exponent e
e = 17  # Common choice: 65537 or 17             # any 1 < e < φ(N) coprime with φ(N); small with few 1-bits (17 = 10001₂) = fast encryption
print(f"\nStep 4: Choose public exponent e")
print(f"  e = {e}")
print(f"  Check: gcd(e, φ(N)) = gcd({e}, {phi_N}) = {math.gcd(e, phi_N)}")
print(f"  Must be 1: {math.gcd(e, phi_N) == 1} ✓")

# Compute private exponent d
d = modular_inverse(e, phi_N)                    # the private exponent, e·d ≡ 1 (mod φ(N)): 2753 (the box below gets it with three divisions)
print(f"\nStep 5: Compute private exponent d")
print(f"  d = {d}")
print(f"  Verify: e×d mod φ(N) = {e}×{d} mod {phi_N} = {(e*d) % phi_N}")
print(f"  Must be 1: {(e*d) % phi_N == 1} ✓")

# Display keys
print("\n" + "="*60)                             # "="*60 repeats the string: a 60-character rule
print("RSA KEY PAIR GENERATED")
print("="*60)
print(f"Public Key:  (e={e}, N={N})")
print(f"Private Key: (d={d}, N={N})")
print("="*60)


Step 1: Choose primes
  p = 61
  q = 53

Step 2: Compute N = p × q
  N = 61 × 53 = 3233

Step 3: Compute φ(N) = (p-1)(q-1)
  φ(3233) = (61-1)×(53-1) = 3120

Step 4: Choose public exponent e
  e = 17
  Check: gcd(e, φ(N)) = gcd(17, 3120) = 1
  Must be 1: True ✓

Step 5: Compute private exponent d
  d = 2753
  Verify: e×d mod φ(N) = 17×2753 mod 3120 = 1
  Must be 1: True ✓

RSA KEY PAIR GENERATED
Public Key:  (e=17, N=3233)
Private Key: (d=2753, N=3233)


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `p, q = 61, 53` | the two secret primes; `N = p * q = 3233` is published, and nobody should be able to split it again |
| `euler_totient(p, q)` | φ(3233) = 60 · 52 = 3120: the secret the attacker needs and cannot get without p and q |
| `e = 17`, `math.gcd(e, phi_N) == 1` | the public exponent must be invertible mod φ(N); 17 (and the standard 65537 = 2¹⁶ + 1) have only two 1-bits, so encryption costs 4 (16) squarings and one multiplication |
| `d = modular_inverse(e, phi_N)` | d = 2753 after 2753 loop iterations: e · d = 46801 = 15 · 3120 + 1 (checked by the `(e*d) % phi_N` line); the box below gets d with three divisions |
| `"="*60`, `"\n" + ...` | string repetition and concatenation: the printed rules and blank lines |

</details>


<details open><summary><b>📐 Working it out</b></summary>

**The private exponent by the extended Euclidean algorithm** (what `modular_inverse` finds by trying all 3120 candidates). Divide $3120$ by $17$ and continue with the remainders:

$$3120 = 183\cdot 17 + 9,\qquad 17 = 1\cdot 9 + 8,\qquad 9 = 1\cdot 8 + 1 .$$

The last non-zero remainder is $1$, so $\gcd(17, 3120) = 1$ and the inverse exists. Now walk back up, writing $1$ as a combination of $3120$ and $17$:

$$1 = 9 - 8 = 9 - (17 - 9) = 2\cdot 9 - 17 = 2\,(3120 - 183\cdot 17) - 17 = 2\cdot 3120 - 367\cdot 17 .$$

Hence $17\cdot(-367) \equiv 1 \pmod{3120}$, and $d = -367 + 3120 = 2753$ ✓ — three divisions instead of 2753 trials (the cell prints `d = 2753`). For a 2048-bit key the brute force would need about $10^{616}$ trials; Euclid needs at most a few thousand divisions (the number of steps is bounded by $5\times$ the number of decimal digits — Lamé's theorem). Task 1 of the In-depth analysis codes this recursion.

</details>


### Step 4: Encryption

In [7]:
# Choose a message to encrypt
message = 123                                    # the plaintext must be an integer smaller than N (a text is first turned into such integers: In-depth task 3)
print(f"Original message: {message}")
print(f"  (Must be less than N={N})")

# Encrypt using public key
ciphertext = encrypt(message, e, N)              # c = m^e mod N: anyone can do this, (e, N) is public
print(f"\nEncrypting with public key (e={e}, N={N}):")
print(f"  c = m^e mod N")
print(f"  c = {message}^{e} mod {N}")
print(f"  c = {ciphertext}")                     # 855 (the box below computes it by repeated squaring)
print(f"\n✓ Encrypted: {message} → {ciphertext}")


Original message: 123
  (Must be less than N=3233)

Encrypting with public key (e=17, N=3233):
  c = m^e mod N
  c = 123^17 mod 3233
  c = 855

✓ Encrypted: 123 → 855


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `message = 123` | the plaintext must be an integer smaller than N; a text is first turned into such integers (In-depth task 3) |
| `encrypt(message, e, N)` | c = mᵉ mod N = 123¹⁷ mod 3233 = 855: anyone can compute it, the key (N, e) is public |
| `{message}^{e}` in an f-string | a caret is just a character here: the printed line mimics the formula, the computation is `pow` |

</details>


<details open><summary><b>📐 Working it out</b></summary>

**Encryption by repeated squaring** — what `pow(123, 17, 3233)` does. $17 = 10001_2 = 16 + 1$, so we need $123^{16}$ and one more multiplication; every product is reduced mod $3233$ at once, so no number ever exceeds $3233^2 \approx 10^7$:

$$123^2 = 15129 = 4\cdot 3233 + 2197 \equiv 2197,\qquad 123^4 \equiv 2197^2 = 4\,826\,809 = 1492\cdot 3233 + 3173 \equiv 3173 \equiv -60,$$
$$123^8 \equiv (-60)^2 = 3600 \equiv 367,\qquad 123^{16} \equiv 367^2 = 134\,689 = 41\cdot 3233 + 2136 \equiv 2136,$$
$$c = 123^{17} \equiv 2136\cdot 123 = 262\,728 = 81\cdot 3233 + 855 \equiv 855 .$$

Five multiplications for an exponent of 17; in general about $2\log_2 e$. The cell prints `c = 855` ✓. Decrypting needs $\log_2 2753 \approx 11$ squarings plus a few multiplications — still instantaneous, but about three times more work than encrypting: the asymmetry behind the choice of a small $e$.

</details>


### Step 5: Decryption

In [8]:
# Decrypt using private key
decrypted = decrypt(ciphertext, d, N)            # m = c^d mod N: only the owner of d can do this
print(f"Decrypting with private key (d={d}, N={N}):")
print(f"  m = c^d mod N")
print(f"  m = {ciphertext}^{d} mod {N}")
print(f"  m = {decrypted}")                      # 123 again: Euler's theorem at work (box below)
print(f"\n✓ Decrypted: {ciphertext} → {decrypted}")

# Verify
print("\n" + "="*60)
if decrypted == message:                         # the round trip must give back the plaintext
    print("SUCCESS! ✓")
    print(f"Original message: {message}")
    print(f"Decrypted message: {decrypted}")
    print("RSA encryption/decryption verified!")
else:
    print("ERROR: Decryption failed")
print("="*60)


Decrypting with private key (d=2753, N=3233):
  m = c^d mod N
  m = 855^2753 mod 3233
  m = 123

✓ Decrypted: 855 → 123

SUCCESS! ✓
Original message: 123
Decrypted message: 123
RSA encryption/decryption verified!


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `decrypt(ciphertext, d, N)` | m = cᵈ mod N = 855²⁷⁵³ mod 3233 = 123: Euler's theorem makes m^{ed} ≡ m because ed ≡ 1 (mod φ(N)) — box below |
| `if decrypted == message: ... else: ...` | the round-trip check; `==` compares values and the branches print the verdict |

</details>


<details open><summary><b>📐 Working it out</b></summary>

**Why $855^{2753} \bmod 3233$ gives back $123$ — Euler's theorem.** $ed = 17\cdot 2753 = 46801 = 1 + 15\cdot 3120 = 1 + 15\,\varphi(N)$, and for $\gcd(m, N) = 1$ Euler's theorem says $m^{\varphi(N)} \equiv 1 \pmod N$. Hence

$$c^{d} = m^{ed} = m\,\big(m^{\varphi(N)}\big)^{15} \equiv m\cdot 1^{15} = m \pmod N .$$

(For $N = pq$ the result holds even when $\gcd(m, N) \ne 1$: check it prime by prime with Fermat's little theorem, as below.)

**Checking the number by hand, without a 2753-fold product.** Work modulo each prime; Fermat's theorem $a^{p-1} \equiv 1 \pmod p$ lets us reduce the exponent modulo $p - 1$:

* mod $61$: $855 = 14\cdot 61 + 1 \equiv 1$, so $855^{2753} \equiv 1$, and indeed $123 = 2\cdot 61 + 1 \equiv 1$ ✓
* mod $53$: $855 = 16\cdot 53 + 7 \equiv 7$ and $2753 = 52\cdot 52 + 49$, so $855^{2753} \equiv 7^{49}$. By squaring: $7^2 = 49 \equiv -4,\ 7^4 \equiv 16,\ 7^8 \equiv 256 \equiv -9,\ 7^{16} \equiv 81 \equiv 28,\ 7^{32} \equiv 784 \equiv -11$; with $49 = 32 + 16 + 1$: $7^{49} \equiv (-11)(28)(7) = -2156 = -(40\cdot 53 + 36) \equiv -36 \equiv 17$, and $123 = 2\cdot 53 + 17 \equiv 17$ ✓

The only number below $3233$ that is $\equiv 1 \pmod{61}$ and $\equiv 17 \pmod{53}$ is $123$ (Chinese remainder theorem) — the cell prints `m = 123` ✓. This prime-by-prime shortcut is also how real RSA implementations speed up decryption by a factor of about 4.

</details>


## 🔬 Security Analysis

### Why RSA Works

In [9]:
print("Mathematical Proof:")
print("="*60)
print(f"e × d = {e} × {d} = {e*d}")
print(f"e × d mod φ(N) = {e*d} mod {phi_N} = {(e*d) % phi_N}")
print(f"\nTherefore: e × d = 1 + k×φ(N)")
print(f"where k = {(e*d - 1) // phi_N}")        # // = integer division: (46801 − 1)/3120 = 15
print("\nBy Euler's theorem:")
print(f"  m^φ(N) ≡ 1 (mod N)")                  # valid when gcd(m, N) = 1; for N = pq the conclusion below holds for every m anyway
print(f"\nSo:")
print(f"  c^d = (m^e)^d = m^(e×d) = m^(1 + k×φ(N))")
print(f"      = m × (m^φ(N))^k ≡ m × 1^k ≡ m (mod N)")
print("\nThis is why decryption recovers the original message!")
print("="*60)


Mathematical Proof:
e × d = 17 × 2753 = 46801
e × d mod φ(N) = 46801 mod 3120 = 1

Therefore: e × d = 1 + k×φ(N)
where k = 15

By Euler's theorem:
  m^φ(N) ≡ 1 (mod N)

So:
  c^d = (m^e)^d = m^(e×d) = m^(1 + k×φ(N))
      = m × (m^φ(N))^k ≡ m × 1^k ≡ m (mod N)

This is why decryption recovers the original message!


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `(e*d - 1) // phi_N` | integer division: k = 15 in e·d = 1 + k·φ(N) |
| the printed chain | c^d = m^{1 + kφ(N)} = m · (m^{φ(N)})^k ≡ m: Euler's theorem (m^{φ(N)} ≡ 1 for gcd(m, N) = 1) turns the big power into 1^k |

</details>


### Classical Security: Breaking RSA

In [10]:
print("To Break RSA, an attacker needs to:")
print("="*60)
print("1. Know: (e, N) [public key]")
print("2. Want: d [private key]")
print("3. Need: φ(N) to compute d")
print("4. Must: Factor N = p × q to find φ(N) = (p-1)(q-1)")
print("\nFactoring is HARD for large N!")
print("="*60)

print(f"\nExample with our small primes:")
print(f"  N = {N}")
print(f"  Trying to factor...")

# Brute force factoring (only works for small N!)
import time
start = time.time()                              # wall-clock time in seconds, to time the loop
for i in range(2, int(N**0.5) + 1):              # trial division: a composite N has a divisor ≤ √N
    if N % i == 0:                               # i divides N
        factor1 = i
        factor2 = N // i                         # // = exact integer quotient
        break                                    # leave the loop at the first divisor found (the smaller prime)
elapsed = time.time() - start

print(f"  Found: {N} = {factor1} × {factor2}")
print(f"  Time: {elapsed*1000:.3f} ms")          # :.3f = three decimals; ×1000 converts seconds to milliseconds
print(f"\nFor RSA-2048 (617 digit number):")
print(f"  Classical factoring: >10^9 years on modern hardware")
print(f"  Quantum (Shor's): ~hours on large-scale quantum computer")


To Break RSA, an attacker needs to:
1. Know: (e, N) [public key]
2. Want: d [private key]
3. Need: φ(N) to compute d
4. Must: Factor N = p × q to find φ(N) = (p-1)(q-1)

Factoring is HARD for large N!

Example with our small primes:
  N = 3233
  Trying to factor...
  Found: 3233 = 53 × 61
  Time: 0.144 ms

For RSA-2048 (617 digit number):
  Classical factoring: >10^9 years on modern hardware
  Quantum (Shor's): ~hours on large-scale quantum computer


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `range(2, int(N**0.5) + 1)` | trial division up to √N: if N = pq with p ≤ q then p ≤ √N, so the loop must meet p; `int()` truncates the float root |
| `if N % i == 0: ... break` | the first divisor found is the smaller prime (53); `break` leaves the loop at once; `N // i` is the cofactor |
| `time.time()` … `elapsed*1000` | wall-clock timing in milliseconds: a fraction of a millisecond for 57 trials |
| why it does not scale | trial division costs √N ≈ 2^{n/2} steps for an n-bit N: 2¹⁰²⁴ for RSA-2048. The best classical algorithm (the number field sieve) is sub-exponential, exp(c·n^{1/3}(log n)^{2/3}), which is where the '> 10⁹ years' comes from; Shor's algorithm is polynomial, ~n³ |

</details>


## ⚛️ Quantum Threat

### Shor's Algorithm Impact

In [11]:
print("CLASSICAL vs QUANTUM FACTORING")
print("="*60)
print("\nProblem: Factor N into p and q")
print("\nClassical (Best Known):")
print("  Algorithm: General Number Field Sieve (GNFS)")
print("  Complexity: O(exp(∛(ln N ln ln N)))")
print("  Type: Sub-exponential, but still exponential for large N")
print("\nQuantum (Shor's Algorithm):")
print("  Complexity: O((ln N)³)")
print("  Type: Polynomial")
print("  Speedup: EXPONENTIAL")
print("\nExample: Factor RSA-2048 (617 digits)")
print("  Classical: ~10^9 years")
print("  Quantum:   ~hours (requires ~20 million noisy qubits)")
print("\nCurrent Status (2025):")
print("  Largest quantum computers: ~1,000 qubits")
print("  Largest factored: Small test numbers (N < 10^6)")
print("  Timeline to threat: 10-20 years (estimate)")
print("="*60)

CLASSICAL vs QUANTUM FACTORING

Problem: Factor N into p and q

Classical (Best Known):
  Algorithm: General Number Field Sieve (GNFS)
  Complexity: O(exp(∛(ln N ln ln N)))
  Type: Sub-exponential, but still exponential for large N

Quantum (Shor's Algorithm):
  Complexity: O((ln N)³)
  Type: Polynomial
  Speedup: EXPONENTIAL

Example: Factor RSA-2048 (617 digits)
  Classical: ~10^9 years
  Quantum:   ~hours (requires ~20 million noisy qubits)

Current Status (2025):
  Largest quantum computers: ~1,000 qubits
  Largest factored: Small test numbers (N < 10^6)
  Timeline to threat: 10-20 years (estimate)


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `print(...)` only | a text cell in disguise: no computation. The figures are 2019–2025 estimates: Gidney & Ekerå (2019) needed ~20 million noisy qubits for 8 hours; Gidney's 2025 update brings RSA-2048 under one million noisy qubits in under a week — the estimates shrink faster than the hardware grows |
| '~1,000 qubits' | today's largest superconducting chips are physical, noisy qubits; the estimates above count *physical* qubits too, but assume error rates and connectivity not yet reached |

</details>


## 🛡️ Post-Quantum Cryptography

In [12]:
print("POST-QUANTUM CRYPTOGRAPHY ALTERNATIVES")
print("="*60)
print("\nNIST Post-Quantum Standards (2024):")
print("\n1. CRYSTALS-Kyber")
print("   Type: Lattice-based encryption")
print("   Use: Key encapsulation mechanism (KEM)")
print("\n2. CRYSTALS-Dilithium")
print("   Type: Lattice-based signatures")
print("   Use: Digital signatures")
print("\n3. SPHINCS+")
print("   Type: Hash-based signatures")
print("   Use: Digital signatures")
print("\nKey Features:")
print("  ✓ Resistant to quantum attacks")
print("  ✓ Based on hard mathematical problems")
print("  ✓ Being deployed NOW (before quantum threat)")
print("\nTransition Strategy:")
print("  1. Hybrid approach: Classical + Post-quantum")
print("  2. Gradual migration of systems")
print("  3. 'Harvest now, decrypt later' threat")
print("="*60)

POST-QUANTUM CRYPTOGRAPHY ALTERNATIVES

NIST Post-Quantum Standards (2024):

1. CRYSTALS-Kyber
   Type: Lattice-based encryption
   Use: Key encapsulation mechanism (KEM)

2. CRYSTALS-Dilithium
   Type: Lattice-based signatures
   Use: Digital signatures

3. SPHINCS+
   Type: Hash-based signatures
   Use: Digital signatures

Key Features:
  ✓ Resistant to quantum attacks
  ✓ Based on hard mathematical problems
  ✓ Being deployed NOW (before quantum threat)

Transition Strategy:
  1. Hybrid approach: Classical + Post-quantum
  2. Gradual migration of systems
  3. 'Harvest now, decrypt later' threat


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `print(...)` only | no computation; the three schemes became NIST standards in August 2024 under their final names: ML-KEM (FIPS 203, from Kyber), ML-DSA (FIPS 204, from Dilithium) and SLH-DSA (FIPS 205, from SPHINCS+) |
| 'Harvest now, decrypt later' | the reason for migrating before a quantum computer exists: traffic recorded today can be decrypted the day Shor's algorithm runs at scale |

</details>


## 🎓 Exercises

### Exercise 1: RSA with Different Parameters (Easy)
Generate RSA keys with p=17, q=19. Encrypt and decrypt the message m=42.

```python
# Your solution here
p = 17
q = 19
# Continue...
```

### Exercise 2: RSA Security (Medium)
Write a function to brute-force factor N given only the public key (e, N). Test with N=221.

**Hint**: Try all divisors from 2 to √N

### Exercise 3: Key Size Analysis (Hard)
Calculate how many qubits would be needed to run Shor's algorithm to factor:
1. RSA-1024 (309 digits)
2. RSA-2048 (617 digits)
3. RSA-4096 (1234 digits)

**Hint**: Approximately 2n qubits needed to factor an n-bit number

## 🔬 In-depth analysis

1. **A faster inverse.** `modular_inverse` tries every $x$ below $\varphi(N)$: 2753 steps here, about $10^{616}$ for a real 2048-bit key. Write `egcd(a, b)` (the extended Euclidean algorithm, returning $g, x, y$ with $ax + by = g$), derive $d$ from it, compare with the author's function and with Python's built-in `pow(e, -1, phi_N)`, and time the three.
2. **A real-size exponent.** Take $p = 1\,000\,003$ and $q = 1\,000\,033$ (both prime) with the standard $e = 65537$; encrypt and decrypt $m = 123\,456$ with the notebook's `encrypt` / `decrypt`. Why would `modular_inverse` be hopeless here, while `egcd` and `pow(e, -1, φ)` are instantaneous?
3. **Text.** Encrypt the string `"QC"` by turning it into an integer with `int.from_bytes(b"QC", "big")` and back with `.to_bytes(2, "big")` (with the key of item 2, whose $N$ is larger than any 2-byte integer). What goes wrong with a longer text such as `"Qiskit"`, and what does that tell you about how RSA is used in practice?
4. **Low exponent.** With $p = 101$, $q = 113$ and $e = 3$ (check $\gcd(3, \varphi(N)) = 1$), encrypt $m = 20$. Since $m^3 < N$, the ciphertext is $m^3$ *without* any modular reduction: recover $m$ from $c$ alone with an integer cube root — the attack that makes textbook RSA with $e = 3$ unsafe for short messages, and the reason real systems pad the message before encrypting.

Write your code in the empty cell below; a reference solution follows it. Try first, then compare.


In [13]:
# 🔬 In-depth analysis — your code here



In [14]:
# Reference solution
import time

# (1) extended Euclid: g = gcd(a, b) and the Bézout coefficients x, y with a*x + b*y = g
def egcd(a, b):
    if b == 0: return a, 1, 0                                 # gcd(a, 0) = a = a*1 + 0*0
    g, x, y = egcd(b, a % b)                                  # recursion on (b, a mod b): the divisions of the box, then the walk back up
    return g, y, x - (a // b) * y

t0 = time.perf_counter(); d_brute = modular_inverse(e, phi_N); t1 = time.perf_counter()   # perf_counter: a high-resolution clock
g, x, y = egcd(e, phi_N); d_euclid = x % phi_N; t2 = time.perf_counter()                 # x may be negative (−367): % brings it into 0 … φ−1
d_pow = pow(e, -1, phi_N); t3 = time.perf_counter()                                      # Python ≥ 3.8: a negative exponent in 3-argument pow = modular inverse
print(f"(1) egcd: {e}*({x}) + {phi_N}*({y}) = {g}  ->  d = {d_euclid}; brute force gives {d_brute}, pow(e, -1, phi_N) gives {d_pow}")
print(f"    time: brute force {1e6 * (t1 - t0):.0f} µs, egcd {1e6 * (t2 - t1):.1f} µs, pow {1e6 * (t3 - t2):.1f} µs")

# (2) a key with million-sized primes and the standard exponent
p2, q2, e2 = 1_000_003, 1_000_033, 65537                      # underscores are digit separators, allowed in Python literals
N2 = p2 * q2; phi2 = euler_totient(p2, q2); d2 = pow(e2, -1, phi2)
m2 = 123_456; c2 = encrypt(m2, e2, N2)
print(f"(2) N = {N2}, phi = {phi2}, d = {d2}; c = {c2}, decrypted = {decrypt(c2, d2, N2)}")
print(f"    modular_inverse would loop d = {d2} times: about {d2 / 1e7 / 3600:.0f} hours at 10^7 iterations/s; egcd needs ~{len(str(phi2)) * 5} divisions at most")

# (3) text -> integer -> cipher -> integer -> text
m3 = int.from_bytes(b"QC", "big")                              # "big" = most significant byte first: ord('Q')*256 + ord('C')
c3 = encrypt(m3, e2, N2); back = decrypt(c3, d2, N2).to_bytes(2, "big").decode()
print(f"(3) 'QC' -> {m3} -> cipher {c3} -> {back!r}")           # !r = repr(): the string is printed with its quotes
m_long = int.from_bytes(b"Qiskit", "big")                      # 6 bytes: up to 2^48 ≈ 2.8e14, larger than N2 ≈ 1.0e12
print(f"    'Qiskit' -> {m_long} > N = {N2}: {m_long > N2}; decrypt(encrypt(m)) = {decrypt(encrypt(m_long, e2, N2), d2, N2)} = m mod N, the message is lost"
      " -> RSA encrypts only blocks smaller than N (in practice: a symmetric key, not the text itself)")

# (4) e = 3 on a small message: the cube root attack
p4, q4, e4 = 101, 113, 3; N4 = p4 * q4; phi4 = euler_totient(p4, q4)
assert math.gcd(e4, phi4) == 1                                  # 3 must be invertible mod φ = 11200 = 2^6 · 5^2 · 7
m4 = 20; c4 = encrypt(m4, e4, N4)                               # m^3 = 8000 < N = 11413: no reduction happened
root = round(c4 ** (1 / 3)); root = next(r for r in (root - 1, root, root + 1) if r ** 3 == c4)   # exact integer cube root: check the neighbours of the float estimate
print(f"(4) e = 3: c = {c4} = m^3 exactly, cube root = {root} = m  (no private key needed)")


(1) egcd: 17*(-367) + 3120*(2) = 1  ->  d = 2753; brute force gives 2753, pow(e, -1, phi_N) gives 2753
    time: brute force 156 µs, egcd 48.8 µs, pow 31.3 µs
(2) N = 1000036000099, phi = 1000034000064, d = 983264276609; c = 915632893244, decrypted = 123456
    modular_inverse would loop d = 983264276609 times: about 27 hours at 10^7 iterations/s; egcd needs ~65 divisions at most
(3) 'QC' -> 20803 -> cipher 896582680593 -> 'QC'
    'Qiskit' -> 89513349835124 > N = 1000036000099: True; decrypt(encrypt(m)) = 510145826313 = m mod N, the message is lost -> RSA encrypts only blocks smaller than N (in practice: a symmetric key, not the text itself)
(4) e = 3: c = 8000 = m^3 exactly, cube root = 20 = m  (no private key needed)


<details open><summary><b>🔧 Reading the code</b></summary>

| construct | what it does |
|---|---|
| `egcd(b, a % b)` | Euclid's recursion: gcd(a, b) = gcd(b, a mod b); unwinding it rebuilds the Bézout coefficients, exactly the 'walk back up' of the box |
| `pow(e, -1, phi_N)` | Python ≥ 3.8 computes modular inverses natively (by the same algorithm): the reference for our `egcd` |
| `time.perf_counter()` | a high-resolution clock for timing short code: tens of microseconds for Euclid, about a tenth of a millisecond for the 2753 brute-force trials — a modest ratio at toy size that explodes with the key length |
| `1_000_003` | underscores as digit separators in numeric literals |
| `int.from_bytes(b, "big")`, `.to_bytes(2, "big")` | bytes ↔ integer: the standard bridge between text and the integers RSA works on; a message larger than N comes back reduced mod N, i.e. destroyed |
| `round(c ** (1/3))` + neighbours | a float cube root is only approximate: the integer check on r−1, r, r+1 makes it exact |

</details>


## 📖 References

### Academic Papers

1. **RSA Original Paper**:  
   Rivest, R.L., Shamir, A., & Adleman, L. (1978).  
   *A method for obtaining digital signatures and public-key cryptosystems*.  
   Communications of the ACM, 21(2), 120-126.

2. **Shor's Algorithm**:  
   Shor, P. W. (1997).  
   *Polynomial-time algorithms for prime factorization and discrete logarithms on a quantum computer*.  
   SIAM Journal on Computing, 26(5), 1484-1509.

3. **Post-Quantum Cryptography**:  
   NIST (2024). *Post-Quantum Cryptography Standardization*.  
   https://csrc.nist.gov/projects/post-quantum-cryptography

### Online Resources

- [RSA Explained](https://www.comparitech.com/blog/information-security/rsa-encryption/)
- [Qiskit Textbook - Shor's Algorithm](https://learn.qiskit.org/course/ch-algorithms/shors-algorithm)
- [Post-Quantum Cryptography FAQ](https://pqshield.com/pqc-101/)

### Related Notebooks

- `Shors_Algorithm_v2.2_enhanced.ipynb`
- `Period_Finding_v2.2_enhanced.ipynb`
- `Quantum_Phase_Estimation_v2.2_enhanced.ipynb`

---

**Course**: Quantum Computing Laboratory  
**License**: MIT  
**Version**: 2.2.0  
**Last Updated**: November 2025